In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [3]:
dim_proveedor=pd.read_sql_table('dim_proveedor', etl_conn)
dim_mensajero=pd.read_sql_table('dim_mensajero', etl_conn)
dim_sede=pd.read_sql_table('dim_sede', etl_conn)
dim_hora=pd.read_sql_table('dim_hora', etl_conn)    
dim_geografia=pd.read_sql_table('dim_geografia', etl_conn)
dim_fecha=pd.read_sql_table('dim_fecha', etl_conn)
dim_usuario=pd.read_sql_table('dim_usuario', etl_conn)
trans_servicio=pd.read_sql_table('trans_servicio', etl_conn)

tabla_sede=pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)
tabla_estados=pd.read_sql_table('mensajeria_estadosservicio', mensajeria)

In [4]:
trans_servicio.head()


,servicio_id,fecha_solicitud,hora_solicitud,cliente_id,mensajero_id,usuario_id,ciudad_destino_id,ciudad_origen_id,mensajero2_id,mensajero3_id,...,hora_mensajero_asignado,hora_recogido_mensajero,hora_entregado,hora_finalizado_completo,minutos_solicitud_iniciado,minutos_iniciado_asignado,minutos_asignado_recogido,minutos_recogido_entregado,minutos_entregado_cerrado,minutos_totales_servicio
0,34,2023-10-26,09:46:03,5,0,10,1,1,0,0,...,None,None,None,None,0.0,NaN,NaN,NaN,NaN,NaN
1,35,2023-10-26,11:18:14,5,7,8,1,1,0,0,...,14:43:08,19:45:18,01:25:34,None,0.0,3084.9,302.166667,56500.266667,NaN,NaN
2,36,2023-10-28,19:21:01,5,0,8,1,1,0,0,...,None,None,None,None,0.0,NaN,NaN,NaN,NaN,NaN
3,41,2023-11-07,09:46:09,5,0,173,1,1,0,0,...,None,None,None,None,0.0,NaN,NaN,NaN,NaN,NaN
4,42,2023-11-07,09:46:10,5,0,173,1,1,0,0,...,None,None,None,None,0.0,NaN,NaN,NaN,NaN,NaN


In [5]:
hecho_servicios= trans_servicio.merge(dim_proveedor[['id_proveedor','key_proveedor']], left_on='cliente_id',right_on='id_proveedor', how='left') \
    .rename(columns={'key_proveedor':'key_dim_proveedor'}) \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero_id',right_on='id_operaciones', how='left') \
    .rename(columns={'key_mensajero':'key_mensajero_1'}) \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero2_id', right_on='id_operaciones', how='left') \
    .rename(columns={'key_mensajero':'key_mensajero_2'}) \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero3_id', right_on='id_operaciones', how='left')  \
    .rename(columns={'key_mensajero':'key_mensajero_3'}) \
    .merge(dim_sede[['sede_id','key_sede']], left_on='sede_id', right_on='sede_id', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_origen_id',right_on='ciudad_id', how='left') \
    .rename(columns={'key_geografia':'key_geografia_origen'}) \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_destino_id',right_on='ciudad_id', how='left') \
    .rename(columns={'key_geografia':'key_geografia_destino'})

hecho_servicios['key_mensajero_1'].value_counts()

key_mensajero_1
33.0    2439
39.0    1548
18.0    1510
48.0    1456
23.0    1348
47.0    1333
4.0     1327
28.0    1252
2.0     1249
6.0     1228
34.0    1099
50.0    1067
9.0     1059
43.0     920
40.0     917
46.0     843
31.0     732
15.0     726
35.0     684
16.0     621
17.0     604
8.0      562
30.0     558
25.0     436
3.0      394
38.0     185
27.0     179
49.0     164
22.0     137
42.0     128
21.0     127
7.0      120
11.0     112
36.0      94
45.0      91
41.0      87
10.0      78
19.0      73
20.0      64
14.0      59
5.0       30
32.0      12
1.0        2
24.0       1
37.0       1
Name: count, dtype: int64

In [6]:
#####Merge con dim_fecha y dim_hora para cada fecha y hora relevante en hecho_servicios

# --- Bloque: solicitud ---
hecho_servicios = hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_solicitud', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_completa','key_hora']], left_on='hora_solicitud', right_on='hora_completa', how='left')

hecho_servicios = hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_solicitud','key_hora':'key_dim_hora_solicitud'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa','hora_completa'], inplace=True)

# --- Bloque: iniciado ---
hecho_servicios = hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_iniciado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_completa','key_hora']], left_on='hora_iniciado', right_on='hora_completa', how='left')

hecho_servicios = hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_iniciado','key_hora':'key_dim_hora_iniciado'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa','hora_completa'], inplace=True)


# --- Bloque: mensajero_asignado ---
hecho_servicios = hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_mensajero_asignado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_completa','key_hora']], left_on='hora_mensajero_asignado', right_on='hora_completa', how='left')

hecho_servicios = hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_mensajero_asignado','key_hora':'key_dim_hora_mensajero_asignado'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa','hora_completa'], inplace=True)


# --- Bloque: recogido_mensajero ---
hecho_servicios = hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_recogido_mensajero', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_completa','key_hora']], left_on='hora_recogido_mensajero', right_on='hora_completa', how='left')

hecho_servicios = hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_recogido_mensajero','key_hora':'key_dim_hora_recogido_mensajero'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa','hora_completa'], inplace=True)


# --- Bloque: entregado ---
hecho_servicios = hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_entregado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_completa','key_hora']], left_on='hora_entregado', right_on='hora_completa', how='left')

hecho_servicios = hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_entregado','key_hora':'key_dim_hora_entregado'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa','hora_completa'], inplace=True)


# --- Bloque: finalizado_completo ---
hecho_servicios = hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_finalizado_completo', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_completa','key_hora']], left_on='hora_finalizado_completo', right_on='hora_completa', how='left')

hecho_servicios = hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_finalizado_completo','key_hora':'key_dim_hora_finalizado_completo'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa','hora_completa'], inplace=True)


hecho_servicios.drop(columns=['id_proveedor','id_operaciones_x','id_operaciones_y','id_operaciones','ciudad_id_x','ciudad_id_y',
                              'mensajero_id','mensajero2_id','mensajero3_id','ciudad_origen_id','ciudad_destino_id','sede_id','cliente_id','usuario_id',
                                'user_id','hora_entregado','hora_iniciado','servicio_id','fecha_solicitud','fecha_iniciado','hora_solicitud','hora_iniciado','fecha_entregado','fecha_finalizado_completo','hora_finalizado_completo',
                                'fecha_mensajero_asignado','fecha_recogido_mensajero',
                                'hora_mensajero_asignado','hora_recogido_mensajero'
                              ], inplace=True)


hecho_servicios.info()

<class 'pandas.DataFrame'>
RangeIndex: 28383 entries, 0 to 28382
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   key_trans_servicio                 28383 non-null  int64  
 1   minutos_solicitud_iniciado         28382 non-null  float64
 2   minutos_iniciado_asignado          27654 non-null  float64
 3   minutos_asignado_recogido          26956 non-null  float64
 4   minutos_recogido_entregado         26889 non-null  float64
 5   minutos_entregado_cerrado          8270 non-null   float64
 6   minutos_totales_servicio           8272 non-null   float64
 7   key_dim_proveedor                  28383 non-null  int64  
 8   key_mensajero_1                    27656 non-null  float64
 9   key_mensajero_2                    2195 non-null   float64
 10  key_mensajero_3                    278 non-null    float64
 11  key_sede                           3573 non-null   float64
 12  k

In [7]:
columnas_keys_fecha_hora = [
    'key_dim_fecha_solicitud', 'key_dim_hora_solicitud',
    'key_dim_fecha_iniciado', 'key_dim_hora_iniciado',
    'key_dim_fecha_mensajero_asignado', 'key_dim_hora_mensajero_asignado',
    'key_dim_fecha_recogido_mensajero', 'key_dim_hora_recogido_mensajero',
    'key_dim_fecha_entregado', 'key_dim_hora_entregado',
    'key_dim_fecha_finalizado_completo', 'key_dim_hora_finalizado_completo'
]

for col in columnas_keys_fecha_hora:
    hecho_servicios[col] = hecho_servicios[col].astype('Int64')

In [8]:

print(hecho_servicios['key_dim_hora_iniciado'].value_counts())

key_dim_hora_iniciado
163042    7
113050    7
102211    6
90028     6
165139    5
         ..
114001    1
82638     1
105202    1
75250     1
104033    1
Name: count, Length: 21156, dtype: Int64


In [9]:
hecho_servicios.head()

,key_trans_servicio,minutos_solicitud_iniciado,minutos_iniciado_asignado,minutos_asignado_recogido,minutos_recogido_entregado,minutos_entregado_cerrado,minutos_totales_servicio,key_dim_proveedor,key_mensajero_1,key_mensajero_2,...,key_dim_fecha_iniciado,key_dim_hora_iniciado,key_dim_fecha_mensajero_asignado,key_dim_hora_mensajero_asignado,key_dim_fecha_recogido_mensajero,key_dim_hora_recogido_mensajero,key_dim_fecha_entregado,key_dim_hora_entregado,key_dim_fecha_finalizado_completo,key_dim_hora_finalizado_completo
0,1,0.0,NaN,NaN,NaN,NaN,NaN,8,NaN,NaN,...,20231026,94603,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,0.0,3084.9,302.166667,56500.266667,NaN,NaN,8,14.0,NaN,...,20231026,111814,20231028,144308,20231028,194518,20231207,12534,<NA>,<NA>
2,3,0.0,NaN,NaN,NaN,NaN,NaN,8,NaN,NaN,...,20231028,192101,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,4,0.0,NaN,NaN,NaN,NaN,NaN,8,NaN,NaN,...,20231107,94609,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,5,0.0,NaN,NaN,NaN,NaN,NaN,8,NaN,NaN,...,20231107,94610,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [10]:
hecho_servicios['key_dim_fecha_entregado'].value_counts()

key_dim_fecha_entregado
20240827    214
20240830    213
20240821    212
20240517    208
20240828    206
           ... 
20231209      1
20231226      1
20231029      1
20231031      1
20231013      1
Name: count, Length: 242, dtype: Int64

In [11]:
# Compara los sets de valores únicos
  # si esto es 0 o muy bajo, ahí está el problema
# Compara valores "iguales" carácter por carácter
val_a = trans_servicio['hora_iniciado'].iloc[0]
val_b = dim_hora['hora_completa'].iloc[0]
print(repr(val_a), repr(val_b))   # repr muestra comillas, espacios ocultos, tipo exacto
print(type(val_a), type(val_b))

datetime.time(9, 46, 3) datetime.time(0, 0)
<class 'datetime.time'> <class 'datetime.time'>


In [12]:
hecho_servicios.to_sql('hecho_servicios', etl_conn, if_exists='replace', index=False)
print(f'hecho_novedades cargada correctamente: {hecho_servicios.shape[0]} filas, {hecho_servicios.shape[1]} columnas')

hecho_novedades cargada correctamente: 28383 filas, 26 columnas
